In [1]:
from topic_gen.generate import Generator
import os
from topic_gen.models import TRECTopic
from src.data import uqv_parser
from topic_gen import logger
import torch
from langchain_community.llms import VLLM

logger.setLevel("WARNING")

In [11]:
parser = uqv_parser()
topics = parser.parse_variants()

In [3]:
# flush cache
torch.cuda.empty_cache()

In [2]:
print(f"Cuda: {torch.cuda.is_available()}")
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
print(f"Number of GPUs: {torch.cuda.device_count()}")

Cuda: True
Number of GPUs: 1


In [5]:
model_dir = "../data/raw/datasets/cache/"
model_name = "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8"

llm = VLLM(model=model_name, 
          download_dir=model_dir,
          # trust_remote_code=True,
          temperature=0.7,
          vllm_kwargs={
            "max_model_len": 32768, 
            "max_num_batched_tokens": 32768,
            }
)

INFO 08-25 14:54:06 [__init__.py:241] Automatically detected platform cuda.
INFO 08-25 14:54:07 [utils.py:326] non-default args: {'model': 'Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', 'download_dir': '../data/raw/datasets/cache/', 'max_model_len': 32768, 'max_num_batched_tokens': 32768, 'disable_log_stats': True}
INFO 08-25 14:54:14 [__init__.py:711] Resolved architecture: Qwen3MoeForCausalLM
INFO 08-25 14:54:14 [__init__.py:1750] Using max model len 32768
INFO 08-25 14:54:15 [scheduler.py:222] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 08-25 14:54:20 [__init__.py:241] Automatically detected platform cuda.
(EngineCore_0 pid=7688) INFO 08-25 14:54:22 [core.py:636] Waiting for init message from front-end.
(EngineCore_0 pid=7688) INFO 08-25 14:54:22 [core.py:74] Initializing a V1 LLM engine (v0.10.1.1) with config: model='Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', speculative_config=None, tokenizer='Qwen/Qwen3-30B-A3B-Instruct-2507-FP8', skip_tokenizer_init=False, tokenizer

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:02<00:06,  2.20s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:02<00:02,  1.24s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:05<00:01,  1.70s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:07<00:00,  2.02s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:07<00:00,  1.89s/it]
(EngineCore_0 pid=7688) 


(EngineCore_0 pid=7688) INFO 08-25 14:54:32 [default_loader.py:262] Loading weights took 7.92 seconds
(EngineCore_0 pid=7688) WARNING 08-25 14:54:32 [marlin_utils_fp8.py:80] Your GPU does not have native support for FP8 computation but FP8 quantization is being used. Weight-only FP8 compression will be used leveraging the Marlin kernel. This may degrade performance for compute-heavy workloads.
(EngineCore_0 pid=7688) INFO 08-25 14:54:34 [gpu_model_runner.py:2007] Model loading took 29.5422 GiB and 10.098786 seconds
(EngineCore_0 pid=7688) INFO 08-25 14:54:44 [backends.py:548] Using cache directory: /home/vscode/.cache/vllm/torch_compile_cache/631635ac22/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_0 pid=7688) INFO 08-25 14:54:44 [backends.py:559] Dynamo bytecode transform time: 9.43 s
(EngineCore_0 pid=7688) INFO 08-25 14:54:52 [backends.py:161] Directly load the compiled graph(s) for dynamic shape from the cache, took 8.093 s
(EngineCore_0 pid=7688) INFO 08-25 14:54:53 [marl

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:07<00:00,  8.40it/s]


(EngineCore_0 pid=7688) INFO 08-25 14:55:06 [gpu_model_runner.py:2708] Graph capturing finished in 9 secs, took 1.09 GiB
(EngineCore_0 pid=7688) INFO 08-25 14:55:06 [core.py:214] init engine (profile, create kv cache, warmup model) took 32.44 seconds
INFO 08-25 14:55:07 [llm.py:298] Supported_tasks: ['generate']


In [6]:
n_queries = 1
n_docs = 1
for topic in topics[200:210]:
    queries_test = []
    docs_test = []
    qids = []
    
    try:
        queries_test.append("\n".join(topic["uqv"][:n_queries]))
        docs = topic["rel_docs"][:n_docs]
        docs_truncated = [doc[:10000] for doc in docs] # truncate to the first 10000 characters
        docs_test.append("\n\n".join(docs_truncated))

        qids.append(topic["qid"])
        
    except Exception as e:
        logger.error(f"Error processing topic {topic['qid']}: {e}")

# generate
prompt_name = "trec-base"
generator = Generator(llm=llm, output_class=TRECTopic, prompt_name=prompt_name)

output = generator.generate(
    queries=queries_test,
    relevant_documents=docs_test,
    config={"max_concurrency": 10},
    item_ids=qids
)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [7]:
from topic_gen.models import Topics

In [14]:
for n_queries in range(1,10):
    for n_docs in range(1, 10):
        # prepare data
        queries_test = []
        docs_test = []
        qids = []
        for topic in topics[200:210]:
            try:
                queries_test.append("\n".join(topic["uqv"][:n_queries]))
                docs = topic["rel_docs"][:n_docs]
                docs_truncated = [doc[:10000] for doc in docs] # truncate to the first 10000 characters
                docs_test.append("\n\n".join(docs_truncated))

                qids.append(topic["qid"])
                
            except Exception as e:
                logger.error(f"Error processing topic {topic['qid']}: {e}")
        
        # generate
        prompt_name = "trec-base"
        generator = Generator(llm=llm, output_class=TRECTopic, prompt_name=prompt_name)

        output = generator.generate(
            queries=queries_test,
            relevant_documents=docs_test,
            item_ids=qids,
            config={"max_concurrency": 10}
        )
        
        # save
        base_dir = "../data/processed/"

        generated_topics = Topics[TRECTopic](topics=output)
        generated_topics.model_dump_jsonl(os.path.join(base_dir, "results", f"topics-robust-{model_name.replace('/', '')}-{prompt_name}-q{n_queries}-d{n_docs}.jsonl"))

Adding requests:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/10 [00:00<?, ?it/s]

...

Adding requests:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Adding requests:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

ValidationError: 1 validation error for Topics[TRECTopic]
topics.1
  Input should be a valid dictionary or instance of TRECTopic [type=model_type, input_value=OutputParserException('In...UTPUT_PARSING_FAILURE '), input_type=OutputParserException]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type